In [ ]:
import os
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# =====================================================================
# 1. SETUP CONFIGURATION
# =====================================================================
# Tentukan folder tujuan (WAJIB pakai PATH ABSOLUT agar Chrome tidak bingung)
download_dir = os.path.abspath("data/downloaded_pdf")
os.makedirs(download_dir, exist_ok=True)

# Konfigurasi Chrome Options agar otomatis download ke folder proyek
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")

# Trik inti: Mengatur preferensi internal browser Chrome agar langsung download PDF
prefs = {
    "download.default_directory": download_dir, 
    "download.prompt_for_download": False,       
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True   
}
options.add_experimental_option("prefs", prefs)

# Jalankan browser Chrome
print("Membuka browser otomatis...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

BASE_URL = "https://putusan.mahkamahagung.go.id/pencarian.html"
target_links = [] # Tempat menyimpan link yang ditemukan
max_pages = 3      # Jumlah halaman pencarian yang mau disisir

# =====================================================================
# 2. PROSES PENCARIAN LINK DETAIL PUTUSAN
# =====================================================================
try:
    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}?q=pembunuhan+pasal+340&page={page}"
        print(f"Mencari link di halaman {page} lewat Selenium...")
        
        driver.get(url)
        time.sleep(5) # Jeda waktu agar JavaScript selesai loading
        
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        
        for a_tag in soup.find_all("a", href=True):
            if "/putusan/" in a_tag["href"] and a_tag["href"] not in target_links:
                target_links.append(a_tag["href"])

    print(f"\nTotal link detail putusan yang ditemukan: {len(target_links)}")

# =====================================================================
# 3. PROSES DOWNLOAD PDF SECARA OTOMATIS
# =====================================================================
    if len(target_links) > 0:
        print(f"Mulai mengunduh dokumen ke folder: {download_dir}\n")
        downloaded_count = 0
        
        for idx, url in enumerate(target_links):
            if downloaded_count >= 30: # Batasi minimal 30 dokumen sesuai ketentuan tugas
                print("\nTarget 30 dokumen putusan pembunuhan sudah terpenuhi!")
                break
                
            print(f"[{downloaded_count+1}/30] Mengunjungi halaman detail: {url}")
            driver.get(url)
            time.sleep(4) 
            
            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")
            
            pdf_link = None
            for a in soup.find_all("a", href=True):
                # PERBAIKAN: Deteksi folder /pdf/ atau /download_file/ sesuai hasil inspect element
                if "/pdf/" in a["href"] or "/download_file/" in a["href"]:
                    pdf_link = a["href"]
                    break
            
            if pdf_link:
                print("   -> Link PDF ditemukan! Memicu download...")
                driver.get(pdf_link)
                time.sleep(5) # Jeda waktu agar proses download file selesai
                downloaded_count += 1
            else:
                print("   -> Tombol file PDF tidak ditemukan. Lanjut ke link berikutnya.")
    else:
        print("Tidak ada link yang ditemukan. Silakan periksa kembali query pencarian Anda.")

except Exception as e:
    print(f"\nTerjadi kendala saat eksekusi: {e}")

finally:
    # Selalu tutup browser di akhir proses agar RAM tidak bengkak
    print("\nMenutup browser otomatis...")
    driver.quit()

print("\n=======================================================")
print(f"Selesai! Silakan cek folder kamu di: {download_dir}")
print("=======================================================")

Membuka browser otomatis...
Mencari link di halaman 1 lewat Selenium...
Mencari link di halaman 2 lewat Selenium...
Mencari link di halaman 3 lewat Selenium...

Total link detail putusan yang ditemukan: 82
Mulai mengunduh dokumen ke folder: d:\Kuliah\Semester 6\Computer Reasoning\Sub-CPMK3\cbr-pembunuhan\notebooks\data\downloaded_pdf

[1/30] Mengunjungi halaman detail: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf00854f71b6584bfeb303930393036.html
   -> Tombol file PDF tidak ditemukan. Lanjut ke link berikutnya.
[1/30] Mengunjungi halaman detail: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf009221056d1468ea8303933373135.html
   -> Tombol file PDF tidak ditemukan. Lanjut ke link berikutnya.
[1/30] Mengunjungi halaman detail: https://putusan3.mahkamahagung.go.id/direktori/putusan/11f03b9bdccdd430b66c313531353035.html
   -> Tombol file PDF tidak ditemukan. Lanjut ke link berikutnya.
[1/30] Mengunjungi halaman detail: https://putusan3.mahkamahagung.go.id/direktori

In [1]:
import os
import re
from pypdf import PdfReader  # Pastikan sudah install: pip install pypdf

# Tentukan jalur folder input dan output
PDF_FOLDER = os.path.abspath("data/downloaded_pdf")
RAW_TXT_FOLDER = os.path.abspath("data/raw")
os.makedirs(RAW_TXT_FOLDER, exist_ok=True)

def clean_court_text(text):
    """
    Fungsi untuk membersihkan noise teks pada putusan pengadilan
    """
    # 1. Mengubah teks menjadi lowercase (huruf kecil semua)
    text = text.lower()
    
    # 2. Menghapus header & footer standar Mahkamah Agung / Pengadilan Negeri
    text = re.sub(r"direktori putusan mahkamah agung republik indonesia.*?\n", "", text)
    text = re.sub(r"halaman \d+ dari \d+ hal\..*?\n", "", text)
    text = re.sub(r"pengadilan negeri manado.*?\n", "", text)
    
    # 3. Menghapus nomor halaman sebatang kara yang sering muncul di sela teks
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)
    
    # 4. Normalisasi spasi ganda, tab, atau baris baru berlebih menjadi satu spasi biasa
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def run_cleaning_pipeline():
    # Ambil semua file yang berakhiran .pdf di folder download
    pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.endswith('.pdf')]
    
    if not pdf_files:
        print(f"Folder {PDF_FOLDER} kosong. Pindahkan file PDF hasil download manualmu ke sana dahulu!")
        return

    print(f"Ditemukan {len(pdf_files)} file PDF. Memulai proses ekstraksi & pembersihan...")
    
    success_count = 0
    for idx, filename in enumerate(sorted(pdf_files)):
        pdf_path = os.path.join(PDF_FOLDER, filename)
        
        # Format penamaan file output harus terstruktur (misal: case 001.txt)
        output_filename = f"case_{idx+1:03d}.txt"
        output_path = os.path.join(RAW_TXT_FOLDER, output_filename)
        
        try:
            # Membaca file PDF
            reader = PdfReader(pdf_path)
            full_text = ""
            
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    full_text += page_text + "\n"
            
            # Bersihkan teks hasil ekstraksi
            cleaned_text = clean_court_text(full_text)
            
            # Validasi keutuhan teks sebelum disimpan (minimal ada isinya)
            if len(cleaned_text) > 500:
                with open(output_path, 'w', encoding='utf-8') as f:
                    f.write(cleaned_text)
                print(f"[{idx+1}] Berhasil mengonversi: {filename} -> {output_filename}")
                success_count += 1
            else:
                print(f"[{idx+1}] Peringatan: Isi teks {filename} terlalu pendek (kemungkinan hasil scan gambar/corrupt).")
                
        except Exception as e:
            print(f"[{idx+1}] Gagal memproses {filename}. Error: {e}")
            
    print(f"\n=======================================================")
    print(f"Selesai! {success_count} file teks bersih tersimpan di: {RAW_TXT_FOLDER}")
    print(f"=======================================================")

# Jalankan pipeline pembersihan
run_cleaning_pipeline()

Ditemukan 40 file PDF. Memulai proses ekstraksi & pembersihan...
[1] Berhasil mengonversi: putusan_100_pid_2023_pt_mnd_20260621125450.pdf -> case_001.txt
[2] Berhasil mengonversi: putusan_100_pid_2023_pt_mnd_20260621130322.pdf -> case_002.txt
[3] Berhasil mengonversi: putusan_102_pid_2024_pt_mnd_20260621123313.pdf -> case_003.txt
[4] Berhasil mengonversi: putusan_103_pid_2020_pt_mnd_20260621140948.pdf -> case_004.txt
[5] Berhasil mengonversi: putusan_106_pid_2022_pt_mnd_20260621140509.pdf -> case_005.txt
[6] Berhasil mengonversi: putusan_110_pid_2024_pt_mnd_20260621123251.pdf -> case_006.txt
[7] Berhasil mengonversi: putusan_112_pid_2025_pt_mnd_20260621122204.pdf -> case_007.txt
[8] Berhasil mengonversi: putusan_115_pid_2023_pt_mnd_20260621123744.pdf -> case_008.txt
[9] Berhasil mengonversi: putusan_122_pid_2025_pt_mnd_20260621122249 (1).pdf -> case_009.txt
[10] Berhasil mengonversi: putusan_122_pid_2025_pt_mnd_20260621122249.pdf -> case_010.txt
[11] Berhasil mengonversi: putusan_154_p